In [8]:
!pip install mlflow
!pip install boto3
!pip install awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from transformers import DataCollatorWithPadding
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.pytorch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [11]:
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
MODEL_NAME = "bert-base-uncased"

MAX_LENGTH = 128

BATCH_SIZE = 16

LEARNING_RATE = 2e-5

EPOCHS = 5

WEIGHT_DECAY = 0.01



In [14]:
df= pd.read_csv("/kaggle/input/datasets/bjdhdhjdbd/twitter-data-sentiment/clean_data.csv")
df = df.dropna()
df.head()

,clean_text,category
0,family mormon never try explain still stare pu...,1.0
1,buddhism much lot compatible christianity espe...,1.0
2,seriously say thing first get complex explain ...,-1.0
3,learn want teach different focus goal not wrap...,0.0
4,benefit may want read live buddha live christ ...,1.0


In [21]:
encoder = LabelEncoder()

df["category"] = encoder.fit_transform(df["category"])

In [22]:
NUM_CLASSES = df["category"].nunique()

print(NUM_CLASSES)

3


In [23]:
X = df["clean_text"]

y = df["category"]

In [24]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=SEED,

    stratify=y

)
X_train, X_valid, y_train, y_valid = train_test_split(

    X_train,

    y_train,

    test_size=0.1,

    random_state=SEED,

    stratify=y_train

)

In [25]:
X_train = X_train.reset_index(drop=True)
X_valid = X_valid.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_valid = y_valid.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [26]:
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_valid)}")
print(f"Testing samples    : {len(X_test)}")

Training samples   : 143645
Validation samples : 15961
Testing samples    : 39902


In [27]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [28]:
sentence = "I love Natural Language Processing."

encoding = tokenizer(
    sentence
)

print(encoding)

{'input_ids': [101, 1045, 2293, 3019, 2653, 6364, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [29]:
print(encoding["input_ids"])

[101, 1045, 2293, 3019, 2653, 6364, 1012, 102]


In [30]:
tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"]
)

print(tokens)

['[CLS]', 'i', 'love', 'natural', 'language', 'processing', '.', '[SEP]']


In [31]:
encoding = tokenizer(

    sentence,

    padding="max_length",

    truncation=True,

    max_length=MAX_LENGTH,

    return_attention_mask=True,

    return_tensors="pt"

)

In [32]:
print(encoding["input_ids"].shape)

print(encoding["attention_mask"].shape)

torch.Size([1, 128])
torch.Size([1, 128])


In [33]:
print(MAX_LENGTH)

128


In [34]:
class BertDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):

        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, index):

        text = str(self.texts[index])

        label = self.labels[index]

        encoding = self.tokenizer(

            text,

            truncation=True,

            max_length=self.max_length,

            padding=False,

            return_attention_mask=True

        )

        return {

            "input_ids": encoding["input_ids"],

            "attention_mask": encoding["attention_mask"],

            "labels": label

        }

In [ ]:

data_collator = DataCollatorWithPadding(

    tokenizer=tokenizer,

    return_tensors="pt"

)

In [36]:
train_dataset = BertDataset(

    X_train,

    y_train,

    tokenizer,

    MAX_LENGTH

)

valid_dataset = BertDataset(

    X_valid,

    y_valid,

    tokenizer,

    MAX_LENGTH

)

test_dataset = BertDataset(

    X_test,

    y_test,

    tokenizer,

    MAX_LENGTH

)

In [37]:
train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=data_collator,

    pin_memory=True,

    num_workers=2

)

In [38]:
test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,

    pin_memory=True,

    num_workers=2

)

In [39]:
valid_loader = DataLoader(

    valid_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,

    pin_memory=True,

    num_workers=2

)

In [42]:
batch = next(iter(train_loader))
print(batch.keys())

KeysView({'input_ids': tensor([[  101,  3802, 13465, 12502,  5371,  7514,  9152,  2527,  2615, 16913,
          2072,  3813, 14865,  4169, 10470,  2425,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2796,  3166, 28217, 17516,  3775,  6060, 12826, 16913,  2072,
          8398,  2265,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2657, 16913,  2072, 11655,  2290,  2242,  3371, 17812, 13596,
         13331,  2243,  2172,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2952,  2634,  2377,  2878,  2047,  2208,  3260, 21146, 22462,
          3144,  2634,  3233,  4206,  2686,  2373,  6583,  7389,  7265, 16913,
          2072,   102,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2852,  3

In [43]:
print(batch["input_ids"].shape)

print(batch["attention_mask"].shape)

print(batch["labels"].shape)

torch.Size([16, 30])
torch.Size([16, 30])
torch.Size([16])


In [45]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=NUM_CLASSES

)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [46]:
model = model.to(device)
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [47]:
num_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print(f"Trainable Parameters: {num_params:,}")

Trainable Parameters: 109,484,547


In [48]:
optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

In [50]:
total_training_steps = len(train_loader) * EPOCHS
total_training_steps

44890

In [51]:
warmup_steps = int(

    0.1 * total_training_steps

)

In [52]:
scheduler = get_linear_schedule_with_warmup(

    optimizer=optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_training_steps

)

In [53]:
scaler = torch.amp.GradScaler(

    "cuda",

    enabled=torch.cuda.is_available()

)

In [54]:
class EarlyStopping:

    def __init__(

        self,

        patience=3,

        min_delta=0,

        path="best_model.pt"

    ):

        self.patience = patience

        self.min_delta = min_delta

        self.path = path

        self.best_loss = float("inf")

        self.counter = 0

        self.stop = False


    def __call__(

        self,

        val_loss,

        model

    ):

        if val_loss < self.best_loss - self.min_delta:

            self.best_loss = val_loss

            self.counter = 0

            torch.save(

                model.state_dict(),

                self.path

            )

        else:

            self.counter += 1

            print(

                f"EarlyStopping {self.counter}/{self.patience}"

            )

            if self.counter >= self.patience:

                self.stop = True

In [55]:
early_stopping = EarlyStopping(

    patience=3,

    path="best_bert_model.pt"

)
train_losses = []

valid_losses = []

train_accuracies = []

valid_accuracies = []

In [56]:
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    scaler,
    device
):

    model.train()

    total_loss = 0

    total_correct = 0

    total_examples = 0

    progress_bar = tqdm(
        dataloader,
        desc="Training"
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            loss = outputs.loss

            logits = outputs.logits

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        scheduler.step()

        predictions = torch.argmax(

            logits,

            dim=1

        )

        total_correct += (

            predictions == labels

        ).sum().item()

        total_examples += labels.size(0)

        total_loss += loss.item()

        progress_bar.set_postfix(

            loss=f"{loss.item():.4f}"

        )

    epoch_loss = total_loss / len(dataloader)

    epoch_accuracy = total_correct / total_examples

    return epoch_loss, epoch_accuracy

In [57]:
def validate(
    model,
    dataloader,
    device
):

    model.eval()

    total_loss = 0

    total_correct = 0

    total_examples = 0

    with torch.no_grad():

        progress_bar = tqdm(
            dataloader,
            desc="Validation"
        )

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels

            )

            loss = outputs.loss

            logits = outputs.logits

            predictions = torch.argmax(

                logits,

                dim=1

            )

            total_correct += (

                predictions == labels

            ).sum().item()

            total_examples += labels.size(0)

            total_loss += loss.item()

    epoch_loss = total_loss / len(dataloader)

    epoch_accuracy = total_correct / total_examples

    return epoch_loss, epoch_accuracy

In [58]:
best_accuracy = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(

        model,

        train_loader,

        optimizer,

        scheduler,

        scaler,

        device

    )

    valid_loss, valid_acc = validate(

        model,

        valid_loader,

        device

    )

    train_losses.append(train_loss)

    valid_losses.append(valid_loss)

    train_accuracies.append(train_acc)

    valid_accuracies.append(valid_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.4f}")

    print(f"Validation Loss : {valid_loss:.4f}")
    print(f"Validation Accuracy : {valid_acc:.4f}")

    if valid_acc > best_accuracy:

        best_accuracy = valid_acc

        torch.save(

            model.state_dict(),

            "best_bert_model.pt"

        )

        print("Best model saved.")

    early_stopping(

        valid_loss,

        model

    )

    if early_stopping.stop:

        print("Early stopping.")

        break


Epoch 1/5


Training:   0%|          | 0/8978 [00:00<?, ?it/s]

Validation:   0%|          | 0/998 [00:00<?, ?it/s]

Train Loss : 0.4779
Train Accuracy : 0.8124
Validation Loss : 0.3300
Validation Accuracy : 0.8837
Best model saved.

Epoch 2/5


Training:   0%|          | 0/8978 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/998 [00:00<?, ?it/s]

Train Loss : 0.1676
Train Accuracy : 0.9428
Validation Loss : 0.3428
Validation Accuracy : 0.8971
EarlyStopping 2/3

Epoch 5/5


Training:   0%|          | 0/8978 [00:00<?, ?it/s]

Validation:   0%|          | 0/998 [00:00<?, ?it/s]

Train Loss : 0.1105
Train Accuracy : 0.9626
Validation Loss : 0.3928
Validation Accuracy : 0.8967
EarlyStopping 3/3
Early stopping.


In [63]:
import torch
import numpy as np
from tqdm.auto import tqdm

model.eval()

labels = []
predictions = []
probabilities = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Testing"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)

        labels.extend(batch_labels.cpu().numpy())

        predictions.extend(preds.cpu().numpy())

        probabilities.extend(probs.cpu().numpy())

labels = np.array(labels)
predictions = np.array(predictions)
probabilities = np.array(probabilities)

Testing:   0%|          | 0/2494 [00:00<?, ?it/s]

In [59]:
model.load_state_dict(

    torch.load(

        "best_bert_model.pt",

        map_location=device

    )

)

model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [60]:
print(

    f"Best Validation Accuracy: {best_accuracy:.4f}"

)

Best Validation Accuracy: 0.9008


In [61]:
mlflow.set_tracking_uri("http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/")
mlflow.set_experiment("BERT_Text_Classification")

2026/07/27 15:13:14 INFO mlflow.tracking.fluent: Experiment with name 'BERT_Text_Classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://zg-mlflow/5', creation_time=1785165194942, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1785165194942, lifecycle_stage='active', name='BERT_Text_Classification', tags={}, trace_location=None, workspace='default'>

In [ ]:

with mlflow.start_run() as run:


    mlflow.set_tag("Framework", "PyTorch")
    mlflow.set_tag("Model", "BERT")
    mlflow.set_tag("Transformer", MODEL_NAME)
    mlflow.set_tag("Task", "Text Classification")

    mlflow.log_params({

        "model_name": MODEL_NAME,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "num_classes": NUM_CLASSES,
        "optimizer": "AdamW",
        "scheduler": "LinearWarmup",
        "loss_function": "CrossEntropyLoss",
        "dataset_size": len(df)

    })


    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    try:

        auc = roc_auc_score(

            labels,

            probabilities,

            multi_class="ovr"

        )

        mlflow.log_metric(
            "roc_auc",
            auc
        )

    except:

        pass
    report = classification_report(

        labels,

        predictions,

        target_names=encoder.classes_.astype(str),

        output_dict=True

    )

    with open("classification_report.json", "w") as f:

        json.dump(report, f, indent=4)

    mlflow.log_artifact(
        "classification_report.json"
    )

    df.to_csv(
        "dataset.csv",
        index=False
    )

    mlflow.log_artifact(
        "dataset.csv"
    )


    plt.figure(figsize=(8,5))

    plt.plot(
        train_losses,
        label="Train Loss"
    )

    plt.plot(
        valid_losses,
        label="Validation Loss"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.title("Loss Curve")

    plt.legend()

    plt.grid(True)

    plt.savefig("loss_curve.png")

    plt.close()

    mlflow.log_artifact(
        "loss_curve.png"
    )


    plt.figure(figsize=(8,5))

    plt.plot(
        train_accuracies,
        label="Train Accuracy"
    )

    plt.plot(
        valid_accuracies,
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.title("Accuracy Curve")

    plt.legend()

    plt.grid(True)

    plt.savefig("accuracy_curve.png")

    plt.close()

    mlflow.log_artifact(
        "accuracy_curve.png"
    )

    cm = confusion_matrix(
        labels,
        predictions
    )

    plt.figure(figsize=(8,6))

    sns.heatmap(

        cm,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=encoder.classes_,

        yticklabels=encoder.classes_

    )

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.title("Confusion Matrix")

    plt.savefig(
        "confusion_matrix.png"
    )

    plt.close()

    mlflow.log_artifact(
        "confusion_matrix.png"
    )


    mlflow.pytorch.log_model(

        model,

        artifact_path="bert_model",
        serialization_format="pickle"   

    )

    tokenizer.save_pretrained(
        "tokenizer"
    )

    mlflow.log_artifacts(
        "tokenizer",
        artifact_path="tokenizer"
    )
    mlflow.log_artifact(
        "best_bert_model.pt"
    )

    print("Run ID :", run.info.run_id)

2026/07/27 15:25:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/27 15:25:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/27 15:25:27 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/27 15:25:42 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version

Run ID : ba3b5b5167184d66b49108e2ef5d6407
🏃 View run awesome-ray-128 at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/5/runs/ba3b5b5167184d66b49108e2ef5d6407
🧪 View experiment at: http://ec2-13-50-105-122.eu-north-1.compute.amazonaws.com:5000/#/experiments/5
